# Moduł 8: Relationships w FastAPI

**Ćwiczenie:** #9 - Relationships & Joins

---

## Spis treści

1. [Wprowadzenie](#1-wprowadzenie)
2. [Nested Pydantic Schemas](#2-nested-pydantic-schemas)
3. [Eager Loading w API](#3-eager-loading-w-api)
4. [Problem N+1 Queries](#4-problem-n1-queries)
5. [Practical Patterns](#5-practical-patterns)
6. [Performance Tips](#6-performance-tips)
7. [Best Practices](#7-best-practices)
8. [Demo Live Coding](#8-demo-live-coding)
9. [Przygotowanie do ćwiczenia](#9-przygotowanie-do-ćwiczenia)
10. [Podsumowanie](#10-podsumowanie)

---

## 1. Wprowadzenie

### Zakres tego modułu

**Ten moduł skupia się TYLKO na aspektach FastAPI:**
- Zwracanie zagnieżdżonych danych w API responses
- Nested Pydantic schemas
- Eager loading w praktyce (joinedload, selectinload)
- Unikanie N+1 queries problem
- Performance w API endpoints

**Teoria SQLAlchemy Relationships:**
- Zobacz `materials/sqlalchemy_intro/SQLAlchemy ORM - Relationships.ipynb`
- Tam znajdziesz: relationship(), back_populates, foreign keys, cascade, OneToMany, ManyToMany, etc.

### Czego potrzebujesz?

**Wymagana wiedza (z sqlalchemy_intro/):**
- Jak definiować relationship() w modelach
- Czym jest ForeignKey
- OneToMany vs ManyToOne

**Przykład modeli (już znasz):**
```python
class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(primary_key=True)
    username: Mapped[str] = mapped_column(String(50))
    
    # Relationship
    tasks: Mapped[list["Task"]] = relationship(back_populates="owner")

class Task(Base):
    __tablename__ = "tasks"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))
    
    # Relationship
    owner: Mapped["User"] = relationship(back_populates="tasks")
```

### Problem do rozwiązania

**Chcemy w API zwrócić:**
```json
// GET /users/1
{
  "id": 1,
  "username": "john",
  "tasks": [                    // ← Zagnieżdżone dane!
    {"id": 1, "name": "Task 1"},
    {"id": 2, "name": "Task 2"}
  ]
}
```

**Wyzwania FastAPI:**
1. Jak zdefiniować nested Pydantic schema?
2. Jak efektywnie załadować related objects (eager loading)?
3. Jak uniknąć N+1 queries?
4. Jaki response_model użyć?

---

## 2. Nested Pydantic Schemas

### Problem: Zwracanie related objects

**Proste schema (bez relationships):**
```python
class UserResponse(BaseModel):
    id: int
    username: str
    # Brak tasks!
```

**Potrzebujemy nested schema:**

In [ ]:
from pydantic import BaseModel

# 1. Schema dla nested object (Task)
class TaskSummary(BaseModel):
    """Uproszczony task (bez owner)"""
    id: int
    name: str

    class Config:
        from_attributes = True  # Dla ORM models

# 2. Schema z nested list (User z tasks)
class UserWithTasks(BaseModel):
    """User wraz z listą tasków"""
    id: int
    username: str
    tasks: list[TaskSummary]  # ← Zagnieżdżona lista!

    class Config:
        from_attributes = True

### Dwukierunkowe nested schemas

**Task z owner:**

In [ ]:
# Schema dla owner (bez tasks, żeby uniknąć cyklicznej zależności)
class OwnerSummary(BaseModel):
    """Uproszczony user (bez tasks)"""
    id: int
    username: str

    class Config:
        from_attributes = True

# Schema dla task z owner
class TaskWithOwner(BaseModel):
    """Task wraz z właścicielem"""
    id: int
    name: str
    owner: OwnerSummary  # ← Zagnieżdżony obiekt!

    class Config:
        from_attributes = True

**WAŻNE:** Config.from_attributes = True

```python
class Config:
    from_attributes = True  # ✅ Musi być dla ORM models!
```

**Dlaczego?**
- SQLAlchemy models używają atrybutów (user.tasks)
- Pydantic domyślnie oczekuje dict (user['tasks'])
- `from_attributes=True` pozwala Pydantic czytać z atrybutów

**Przed Pydantic v2:**
```python
class Config:
    orm_mode = True  # ⚠️ Stary styl (Pydantic v1)
```

### Unikanie cyklicznych zależności

**Problem:**
```python
# ❌ Nie zadziała - cykliczna zależność!
class User(BaseModel):
    tasks: list[Task]  # Task jeszcze nie zdefiniowany!

class Task(BaseModel):
    owner: User  # User już zdefiniowany, ale ma Task w liście!
```

**Rozwiązanie 1: Uproszczone schemas (recommended)**
```python
# ✅ Dobre - User ma tasks, Task ma tylko basic owner info
class TaskSummary(BaseModel):  # Bez owner!
    id: int
    name: str

class OwnerSummary(BaseModel):  # Bez tasks!
    id: int
    username: str

class UserWithTasks(BaseModel):
    tasks: list[TaskSummary]

class TaskWithOwner(BaseModel):
    owner: OwnerSummary
```

**Rozwiązanie 2: Forward references (advanced)**
```python
# ⚠️ Możliwe, ale skomplikowane
from __future__ import annotations

class User(BaseModel):
    tasks: list['Task']  # Forward reference

class Task(BaseModel):
    owner: User

User.model_rebuild()  # Pydantic v2
```

**Best practice:** Używaj uproszczonych schemas (Rozwiązanie 1)

---

## 3. Eager Loading w API

### Lazy loading (default behavior)

**Problem:**

In [ ]:
from fastapi import FastAPI, Depends
from sqlalchemy import select
from sqlalchemy.orm import Session

app = FastAPI()

@app.get("/users/{user_id}", response_model=UserWithTasks)
def get_user(user_id: int, db: Session = Depends(get_db)):
    """
    ❌ PROBLEM: Lazy loading!
    
    1. Query: SELECT * FROM users WHERE id = 1
    2. Pydantic próbuje dostać user.tasks
    3. Session już zamknięta (po return)!
    4. ERROR: DetachedInstanceError
    """
    user = db.get(User, user_id)
    return user  # Boom! Session zamknięta, nie możemy załadować tasks

**Error:**
```
DetachedInstanceError: Parent instance <User> is not bound to a Session;
lazy load operation of attribute 'tasks' cannot proceed
```

### Rozwiązanie: Eager Loading

**joinedload() - dla one-to-one lub one-to-few**

In [ ]:
from sqlalchemy.orm import joinedload

@app.get("/users/{user_id}", response_model=UserWithTasks)
def get_user_with_tasks(user_id: int, db: Session = Depends(get_db)):
    """
    ✅ Eager loading z joinedload()
    
    Wykonuje LEFT OUTER JOIN:
    SELECT users.*, tasks.* 
    FROM users 
    LEFT OUTER JOIN tasks ON users.id = tasks.user_id
    WHERE users.id = 1
    
    Wszystko w 1 query!
    """
    stmt = select(User).options(joinedload(User.tasks)).where(User.id == user_id)
    user = db.execute(stmt).scalar_one_or_none()
    
    if not user:
        raise HTTPException(404, "User not found")
    
    # user.tasks już załadowane!
    return user

**selectinload() - dla one-to-many (many items)**

In [ ]:
from sqlalchemy.orm import selectinload

@app.get("/users/{user_id}", response_model=UserWithTasks)
def get_user_with_tasks_v2(user_id: int, db: Session = Depends(get_db)):
    """
    ✅ Eager loading z selectinload()
    
    Wykonuje 2 queries:
    1. SELECT * FROM users WHERE id = 1
    2. SELECT * FROM tasks WHERE user_id IN (1)
    
    Lepsze dla dużej liczby related objects
    """
    stmt = select(User).options(selectinload(User.tasks)).where(User.id == user_id)
    user = db.execute(stmt).scalar_one_or_none()
    
    if not user:
        raise HTTPException(404, "User not found")
    
    return user

### joinedload() vs selectinload()

| Feature | joinedload() | selectinload() |
|---------|--------------|----------------|
| **Queries** | 1 query (JOIN) | 2 queries (separate SELECT) |
| **Use case** | One-to-one, one-to-few | One-to-many (many items) |
| **Performance** | Lepsze dla <10 related | Lepsze dla >10 related |
| **SQL** | LEFT OUTER JOIN | SELECT ... WHERE IN (...) |
| **Duplicates** | Może duplikować parent rows | Nie duplikuje |

**Przykład:**
```python
# ✅ joinedload - user ma 1-5 tasks
stmt = select(User).options(joinedload(User.tasks))

# ✅ selectinload - user ma 100+ tasks
stmt = select(User).options(selectinload(User.tasks))
```

---

## 4. Problem N+1 Queries

### Czym jest N+1?

**Scenariusz:** GET /users (lista wszystkich userów z taskami)

**❌ Lazy loading (N+1 problem):**

In [ ]:
@app.get("/users", response_model=list[UserWithTasks])
def get_users_bad(db: Session = Depends(get_db)):
    """
    ❌ N+1 QUERIES!
    
    Jeśli mamy 100 userów:
    1. SELECT * FROM users                (1 query)
    2. SELECT * FROM tasks WHERE user_id=1  (dla user 1)
    3. SELECT * FROM tasks WHERE user_id=2  (dla user 2)
    ...
    101. SELECT * FROM tasks WHERE user_id=100 (dla user 100)
    
    TOTAL: 101 queries! 💀
    """
    users = db.execute(select(User)).scalars().all()
    return users  # Pydantic próbuje dostać user.tasks dla każdego!

**✅ Eager loading (rozwiązanie):**

In [ ]:
@app.get("/users", response_model=list[UserWithTasks])
def get_users_good(db: Session = Depends(get_db)):
    """
    ✅ Eager loading - tylko 2 queries!
    
    1. SELECT * FROM users
    2. SELECT * FROM tasks WHERE user_id IN (1,2,3,...,100)
    
    TOTAL: 2 queries! ✨
    """
    stmt = select(User).options(selectinload(User.tasks))
    users = db.execute(stmt).scalars().all()
    return users

### Wykrywanie N+1 w praktyce

**Metoda 1: echo=True**
```python
# database.py
engine = create_engine(DATABASE_URL, echo=True)  # Loguje SQL queries
```

**Logi przy N+1:**
```sql
SELECT * FROM users
SELECT * FROM tasks WHERE user_id = 1
SELECT * FROM tasks WHERE user_id = 2
SELECT * FROM tasks WHERE user_id = 3
... (wiele identycznych queries!)
```

**Logi z eager loading:**
```sql
SELECT * FROM users
SELECT * FROM tasks WHERE user_id IN (1, 2, 3, ..., 100)
```

**Metoda 2: SQLAlchemy warnings**
```python
import warnings
from sqlalchemy import exc as sa_exc

warnings.simplefilter("error", sa_exc.SAWarning)
# Teraz lazy loading rzuci warning/error
```

---

## 5. Practical Patterns

### Pattern 1: GET /users/{id} z taskami

In [ ]:
from fastapi import HTTPException, status

@app.get("/users/{user_id}", response_model=UserWithTasks)
def get_user_with_tasks(user_id: int, db: Session = Depends(get_db)):
    """
    Pobierz usera wraz z jego taskami
    
    Response:
    {
      "id": 1,
      "username": "john",
      "tasks": [{"id": 1, "name": "Task 1"}, ...]
    }
    """
    stmt = select(User).options(selectinload(User.tasks)).where(User.id == user_id)
    user = db.execute(stmt).scalar_one_or_none()
    
    if not user:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "User not found")
    
    return user

### Pattern 2: GET /tasks/{id} z owner

In [ ]:
@app.get("/tasks/{task_id}", response_model=TaskWithOwner)
def get_task_with_owner(task_id: int, db: Session = Depends(get_db)):
    """
    Pobierz task wraz z właścicielem
    
    Response:
    {
      "id": 1,
      "name": "Task 1",
      "owner": {"id": 1, "username": "john"}
    }
    """
    stmt = select(Task).options(joinedload(Task.owner)).where(Task.id == task_id)
    task = db.execute(stmt).scalar_one_or_none()
    
    if not task:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "Task not found")
    
    return task

### Pattern 3: GET /users/{id}/tasks (sub-resource)

**Czasem lepiej osobny endpoint niż nested data:**

In [ ]:
@app.get("/users/{user_id}/tasks", response_model=list[TaskSummary])
def get_user_tasks(user_id: int, db: Session = Depends(get_db)):
    """
    Pobierz tylko taski danego usera
    
    Zalety:
    - Można filtrować, sortować, paginować taski
    - Nie zwracamy danych usera (mniejszy payload)
    - REST-ful pattern
    """
    # Sprawdź czy user istnieje
    user = db.get(User, user_id)
    if not user:
        raise HTTPException(404, "User not found")
    
    # Pobierz tylko taski
    stmt = select(Task).where(Task.user_id == user_id)
    tasks = db.execute(stmt).scalars().all()
    
    return tasks

### Pattern 4: Filtrowanie nested objects

In [ ]:
@app.get("/users/{user_id}", response_model=UserWithTasks)
def get_user_with_completed_tasks(user_id: int, db: Session = Depends(get_db)):
    """
    User z TYLKO completed tasks
    
    Zakładając że Task ma pole completed: bool
    """
    from sqlalchemy.orm import selectinload
    
    stmt = (
        select(User)
        .options(
            selectinload(User.tasks).where(Task.completed == True)
        )
        .where(User.id == user_id)
    )
    
    user = db.execute(stmt).scalar_one_or_none()
    if not user:
        raise HTTPException(404, "User not found")
    
    return user

---

## 6. Performance Tips

### Tip 1: Wybieraj odpowiedni loader

```python
# ✅ joinedload - dla one-to-one lub małej liczby related
stmt = select(Task).options(joinedload(Task.owner))  # Task → User (1:1)

# ✅ selectinload - dla one-to-many z wieloma rekordami
stmt = select(User).options(selectinload(User.tasks))  # User → Tasks (1:many)
```

### Tip 2: Ładuj tylko potrzebne kolumny

In [ ]:
from sqlalchemy.orm import load_only

# Załaduj tylko wybrane kolumny
stmt = (
    select(User)
    .options(
        load_only(User.id, User.username),  # Tylko id i username
        selectinload(User.tasks).load_only(Task.id, Task.name)  # Tylko id i name z tasks
    )
)
users = db.execute(stmt).scalars().all()

### Tip 3: Używaj różnych schemas dla różnych endpoints

```python
# Endpoint 1: Lista userów - tylko basic info
class UserSummary(BaseModel):
    id: int
    username: str

@app.get("/users", response_model=list[UserSummary])
def get_users(...):
    # Nie ładuj tasks!
    pass

# Endpoint 2: Pojedynczy user - pełne dane z taskami
class UserDetail(BaseModel):
    id: int
    username: str
    email: str
    tasks: list[TaskSummary]

@app.get("/users/{id}", response_model=UserDetail)
def get_user(...):
    # Załaduj tasks
    pass
```

### Tip 4: Pagination dla nested collections

In [ ]:
@app.get("/users/{user_id}/tasks")
def get_user_tasks_paginated(
    user_id: int,
    page: int = 1,
    page_size: int = 10,
    db: Session = Depends(get_db)
):
    """
    Zamiast zwracać user.tasks (wszystkie),
    zwróć paginowane taski
    """
    user = db.get(User, user_id)
    if not user:
        raise HTTPException(404, "User not found")
    
    skip = (page - 1) * page_size
    
    stmt = (
        select(Task)
        .where(Task.user_id == user_id)
        .offset(skip)
        .limit(page_size)
    )
    
    tasks = db.execute(stmt).scalars().all()
    
    # Count total
    total_stmt = select(func.count()).select_from(Task).where(Task.user_id == user_id)
    total = db.execute(total_stmt).scalar()
    
    return {
        "page": page,
        "page_size": page_size,
        "total": total,
        "tasks": tasks
    }

---

## 7. Best Practices

### ✅ Rób tak:

**1. Zawsze używaj eager loading dla nested data:**
```python
# ✅ Dobre
@app.get("/users/{id}", response_model=UserWithTasks)
def get_user(...):
    stmt = select(User).options(selectinload(User.tasks))
    # ...

# ❌ Złe - lazy loading = DetachedInstanceError
@app.get("/users/{id}", response_model=UserWithTasks)
def get_user(...):
    user = db.get(User, user_id)
    return user  # user.tasks nie załadowane!
```

**2. Używaj uproszczonych schemas dla nested objects:**
```python
# ✅ Dobre - bez cyklicznych zależności
class TaskSummary(BaseModel):  # Bez owner
    id: int
    name: str

class UserWithTasks(BaseModel):
    tasks: list[TaskSummary]

# ❌ Unikaj - cykliczne zależności
class Task(BaseModel):
    owner: User  # User ma tasks: list[Task] - cykl!
```

**3. Config.from_attributes = True:**
```python
# ✅ Dobre - dla ORM models
class UserResponse(BaseModel):
    # ...
    class Config:
        from_attributes = True

# ❌ Złe - brak Config
class UserResponse(BaseModel):
    # ...
    # Nie zadziała z SQLAlchemy models!
```

**4. selectinload dla list, joinedload dla single:**
```python
# ✅ Dobre
select(User).options(selectinload(User.tasks))  # 1:many
select(Task).options(joinedload(Task.owner))    # many:1

# ⚠️ Działa, ale mniej efektywne
select(User).options(joinedload(User.tasks))  # JOIN dla wielu = duplikacja rows
```

**5. Osobne endpointy dla sub-resources:**
```python
# ✅ Dobre - RESTful pattern
GET /users/{id}        # User basic info
GET /users/{id}/tasks  # User's tasks (z pagination, filtering)

# ⚠️ Mniej elastyczne
GET /users/{id}  # User + wszystkie tasks (bez paginacji)
```

### ❌ Nie rób tak:

**1. Nie używaj lazy loading w API:**
```python
# ❌ Bardzo złe - N+1 queries
@app.get("/users")
def get_users(...):
    users = db.query(User).all()
    # Pydantic próbuje user.tasks dla każdego -> N queries!
    return users
```

**2. Nie zwracaj wszystkich nested objects bez limitów:**
```python
# ❌ Złe - user może mieć 10000 tasks!
@app.get("/users/{id}")
def get_user(...):
    # Zwraca WSZYSTKIE taski bez limitu
    stmt = select(User).options(selectinload(User.tasks))
    # ...

# ✅ Dobre - osobny endpoint z paginacją
@app.get("/users/{id}/tasks?page=1&size=10")
```

**3. Nie mieszaj różnych loaderów na tej samej relationship:**
```python
# ❌ Złe - konflikt!
stmt = select(User).options(
    joinedload(User.tasks),
    selectinload(User.tasks)  # Który użyć?!
)
```

In [ ]:
from sqlalchemy import String, ForeignKey
from sqlalchemy.orm import Mapped, mapped_column, relationship
from database import Base

class User(Base):
    __tablename__ = "users"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    username: Mapped[str] = mapped_column(String(50), unique=True)
    email: Mapped[str] = mapped_column(String(100))
    
    # Relationship
    tasks: Mapped[list["Task"]] = relationship(back_populates="owner")

class Task(Base):
    __tablename__ = "tasks"
    
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(100))
    user_id: Mapped[int] = mapped_column(ForeignKey("users.id"))
    
    # Relationship
    owner: Mapped["User"] = relationship(back_populates="tasks")

**schemas.py:**

In [ ]:
from pydantic import BaseModel

# Basic schemas (bez relationships)
class UserBase(BaseModel):
    id: int
    username: str
    
    class Config:
        from_attributes = True

class TaskBase(BaseModel):
    id: int
    name: str
    
    class Config:
        from_attributes = True

# Nested schemas
class TaskSummary(BaseModel):
    """Task bez owner (dla UserWithTasks)"""
    id: int
    name: str
    
    class Config:
        from_attributes = True

class OwnerSummary(BaseModel):
    """User bez tasks (dla TaskWithOwner)"""
    id: int
    username: str
    
    class Config:
        from_attributes = True

class UserWithTasks(BaseModel):
    """User z listą tasków"""
    id: int
    username: str
    email: str
    tasks: list[TaskSummary]
    
    class Config:
        from_attributes = True

class TaskWithOwner(BaseModel):
    """Task z właścicielem"""
    id: int
    name: str
    owner: OwnerSummary
    
    class Config:
        from_attributes = True

**main.py:**

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, status
from sqlalchemy import select
from sqlalchemy.orm import Session, selectinload, joinedload
from database import get_db
from models import User, Task
from schemas import UserWithTasks, TaskWithOwner, TaskSummary

app = FastAPI(title="Relationships API")

# GET /users/{id} - User z taskami
@app.get("/users/{user_id}", response_model=UserWithTasks)
def get_user_with_tasks(user_id: int, db: Session = Depends(get_db)):
    """
    Pobierz usera wraz z jego taskami
    
    Eager loading: selectinload(User.tasks)
    """
    stmt = select(User).options(selectinload(User.tasks)).where(User.id == user_id)
    user = db.execute(stmt).scalar_one_or_none()
    
    if not user:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "User not found")
    
    return user

# GET /tasks/{id} - Task z ownerem
@app.get("/tasks/{task_id}", response_model=TaskWithOwner)
def get_task_with_owner(task_id: int, db: Session = Depends(get_db)):
    """
    Pobierz task wraz z właścicielem
    
    Eager loading: joinedload(Task.owner)
    """
    stmt = select(Task).options(joinedload(Task.owner)).where(Task.id == task_id)
    task = db.execute(stmt).scalar_one_or_none()
    
    if not task:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "Task not found")
    
    return task

# GET /users/{id}/tasks - Sub-resource (tylko taski)
@app.get("/users/{user_id}/tasks", response_model=list[TaskSummary])
def get_user_tasks(user_id: int, db: Session = Depends(get_db)):
    """
    Pobierz tylko taski danego usera
    
    RESTful pattern - osobny endpoint dla sub-resource
    """
    # Sprawdź czy user istnieje
    user = db.get(User, user_id)
    if not user:
        raise HTTPException(status.HTTP_404_NOT_FOUND, "User not found")
    
    # Pobierz taski
    stmt = select(Task).where(Task.user_id == user_id)
    tasks = db.execute(stmt).scalars().all()
    
    return tasks

# GET /users - Lista userów (z eager loading, ale bez tasków w response)
@app.get("/users", response_model=list[UserWithTasks])
def get_users(db: Session = Depends(get_db)):
    """
    Pobierz wszystkich userów z taskami
    
    ⚠️ UWAGA: W produkcji użyj paginacji!
    """
    stmt = select(User).options(selectinload(User.tasks))
    users = db.execute(stmt).scalars().all()
    return users

**Test:**
```bash
# Seed data (załóżmy że mamy 2 users i 3 tasks)

# GET /users/1 - User z taskami
curl http://localhost:8000/users/1
# → {
#     "id": 1,
#     "username": "john",
#     "email": "john@example.com",
#     "tasks": [
#       {"id": 1, "name": "Task 1"},
#       {"id": 2, "name": "Task 2"}
#     ]
#   }

# GET /tasks/1 - Task z ownerem
curl http://localhost:8000/tasks/1
# → {
#     "id": 1,
#     "name": "Task 1",
#     "owner": {"id": 1, "username": "john"}
#   }

# GET /users/1/tasks - Tylko taski
curl http://localhost:8000/users/1/tasks
# → [
#     {"id": 1, "name": "Task 1"},
#     {"id": 2, "name": "Task 2"}
#   ]
```

---

## 10. Podsumowanie

### Kluczowe zagadnienia:

1. **Nested Pydantic Schemas** - zwracanie zagnieżdżonych obiektów w API
2. **Config.from_attributes = True** - dla SQLAlchemy models
3. **Eager Loading** - joinedload() vs selectinload()
4. **N+1 Queries Problem** - zawsze używaj eager loading!
5. **Practical Patterns** - GET /users/{id} z taskami, sub-resources
6. **Performance** - wybieraj odpowiedni loader, ładuj tylko potrzebne dane

### FastAPI-specific (nie ma w sqlalchemy_intro/):

```python
# 1. Nested schemas
class UserWithTasks(BaseModel):
    tasks: list[TaskSummary]  # Zagnieżdżona lista
    class Config:
        from_attributes = True  # Musi być!

# 2. Eager loading w endpoint
@app.get("/users/{id}", response_model=UserWithTasks)
def get_user(...):
    stmt = select(User).options(selectinload(User.tasks))
    user = db.execute(stmt).scalar_one_or_none()
    return user  # FastAPI automatycznie serializuje nested data

# 3. Unikanie N+1
@app.get("/users", response_model=list[UserWithTasks])
def get_users(...):
    # ✅ selectinload - tylko 2 queries zamiast N+1
    stmt = select(User).options(selectinload(User.tasks))
    users = db.execute(stmt).scalars().all()
    return users
```

### Teoria SQLAlchemy Relationships:
- `materials/sqlalchemy_intro/SQLAlchemy ORM - Relationships.ipynb`
- Tam: relationship(), back_populates, cascade, foreign keys, etc.

### Następny krok:
Przejdź do **Moduł 09: Authentication & JWT** - zabezpieczanie API